# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pd
# Define the dataset URLcroissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load the dataset metadatadataset = mlc.Dataset(croissant_url)
# Access the dataset metadata (as object attributes)print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

### List All Record Sets
We'll inspect the dataset to see what record sets are present. The record set `@id` is used for referencing them, as required.

### List Fields/Columns in Each Record Set
We'll also list the fields (and their `@id`) for each record set.

In [ ]:
# Inspect available record setsrecord_sets = dataset.metadata.record_setsfor rs in record_sets:    print(f"RecordSet name: {rs.name}, @id: {rs.id}")    print("  Fields:")    for field in rs.fields:        print(f"    - {field.name} (@id: {field.id}) [column: {field.column.id if field.column else 'N/A'}; dataType: {field.data_type}]")    print()

### Preview Records in the First Record Set
Print a few records using the first record set's `@id`.

In [ ]:
# Get the first record setif len(record_sets) == 0:    print('No record sets found in the metadata.')else:    first_rs_id = record_sets[0].id    print(f"First record set @id: {first_rs_id}\nPreviewing some records:")    for i, rec in enumerate(dataset.records(record_set=first_rs_id)):        print(rec)        if i == 2:            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set into pandas DataFrames (using @id)dataframes = {}record_set_ids = [rs.id for rs in record_sets]
for record_set_id in record_set_ids:    records = list(dataset.records(record_set=record_set_id))    if len(records) > 0:        df = pd.DataFrame(records)        dataframes[record_set_id] = df        print(f"Loaded DataFrame for record set {record_set_id} with shape {df.shape}")        print("Columns:", df.columns.tolist())        print(df.head(2))    else:        print(f"No records found in record set {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section applies to one of the main record sets of interest.

In [ ]:
# Pick the main record set for EDA (replace with your main record set's @id as needed)if len(record_set_ids) == 0:    raise ValueError("No record sets detected.")main_record_set_id = record_set_ids[0]df = dataframes[main_record_set_id]
# List numeric fields (using @id, filter for likely numeric columns by dtype if loaded, else by field metadata)import numpy as npnumeric_col_candidates = df.select_dtypes(include=[np.number]).columns.tolist()print(f"Numeric columns in '{main_record_set_id}':", numeric_col_candidates)
# Select a numeric field for filteringif numeric_col_candidates:    numeric_field_id = numeric_col_candidates[0]  # Or pick as needed    threshold = df[numeric_field_id].quantile(0.75)  # use Q3 as example threshold    filtered_df = df[df[numeric_field_id] > threshold].copy()    print(f"Filtered records with {numeric_field_id} > {threshold} (Q3):")    print(filtered_df.head(3))
    # Normalize selected numeric column    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()    print(f"Normalized {numeric_field_id} for filtered records:")    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(3))
    # Try to group by a likely group/categorical column    non_num_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()    group_field = None    for col in non_num_cols:        nunique = df[col].nunique()        if 2 <= nunique <= 10:            group_field = col            break    if group_field:        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()        print(f"Mean {numeric_field_id} grouped by '{group_field}':")        print(grouped_df.head())else:    print("No numeric fields available for EDA in this record set.")

## 5. Visualization
Visualize the distribution of the numeric field for the selected record set. We'll plot a histogram and boxplot for the normalized numeric field if available.

In [ ]:
import matplotlib.pyplot as pltimport seaborn as sns
if len(record_set_ids) and numeric_col_candidates:
    # Plot histogram
    plt.figure(figsize=(10, 4))    sns.histplot(filtered_df[numeric_field_id], kde=True)    plt.title(f"Histogram of {numeric_field_id} in filtered DataFrame")    plt.xlabel(numeric_field_id)    plt.ylabel("Count")    plt.show()
    if f"{numeric_field_id}_normalized" in filtered_df.columns:        plt.figure(figsize=(10, 4))        sns.boxplot(x=filtered_df[f"{numeric_field_id}_normalized"])        plt.title(f"Boxplot of {numeric_field_id} (normalized)")        plt.xlabel(f"{numeric_field_id}_normalized")        plt.show()

## 6. Conclusion
In this notebook, we loaded a Croissant dataset describing protein abundance and related properties from human extracellular vesicles. Using `mlcroissant`, we:
- Explored the schema and record sets (using their `@id` via the schema).
- Loaded data for each record set into DataFrames for inspection.
- Applied simple filtering and normalization to numeric fields, and grouped records for aggregate statistics.
- Visualized the distribution of selected numeric fields.

For deeper analysis, consider domain-specific processing, further cleaning, and more advanced statistical or ML methods.